# Doctor Assistant — MedSAM2 Colab inference server

Runs the existing full-volume FastAPI route on one Colab GPU and exposes it through ngrok. The Mac keeps OHIF/Orthanc only. Use an **L4 or Ampere-or-newer GPU with at least 18 GiB**. The server permits one inference at a time.

In [ ]:
# Configuration — no credentials belong in this cell.
from pathlib import Path

REPO_URL = 'https://github.com/Hamza09Hamza/doctor_assistant.git'
REPO_REF = 'main'
REPO_DIR = Path('/content/doctor_assistant')
MEDSAM2_DIR = Path('/content/MedSAM2')
API_PORT = 8000
DOWNLOAD_ORTHANC_MACOS = True  # One-time FortiGuard-safe relay for this Mac.


In [ ]:
# Fail before setup if this runtime cannot safely host the model.
import psutil
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
system_gib = psutil.virtual_memory().total / 1024**3
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu.name}, compute capability {major}.{minor}, {gpu_gib:.1f} GiB')
print(f'System RAM: {system_gib:.1f} GiB')
assert major >= 8, 'Use L4/A100/Ampere-or-newer: the API uses CUDA bfloat16 autocast.'
assert gpu_gib >= 18, 'At least 18 GiB GPU memory is required for this bounded test.'
assert system_gib >= 10, 'At least 10 GiB system RAM is required.'


In [ ]:
# Clone the pushed application and install only the server/runtime dependencies.
import os
import subprocess
import sys

def run(command, *, cwd=None, env=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)

if REPO_DIR.exists():
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', REPO_REF], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR)
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)])

run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '--quiet',
     '-r', str(REPO_DIR / 'requirements-api.txt'),
     'idc-index==0.12.5', 'scipy>=1.12', 'pyngrok>=7.2'])

if not MEDSAM2_DIR.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/bowang-lab/MedSAM2.git', str(MEDSAM2_DIR)])
install_env = dict(os.environ, SAM2_BUILD_CUDA='0')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(MEDSAM2_DIR)], env=install_env)
print('Application and official MedSAM2 runtime installed.')


In [ ]:
# Download only the CT-lesion checkpoint, then stage the pinned LIDC study/SEG.
import json
from urllib.request import urlretrieve

checkpoint_dir = MEDSAM2_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint = checkpoint_dir / 'MedSAM2_CTLesion.pt'
if not checkpoint.exists():
    urlretrieve(
        'https://huggingface.co/wanglab/MedSAM2/resolve/main/MedSAM2_CTLesion.pt',
        checkpoint,
    )
print(f'Checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MiB)')

run([sys.executable, 'scripts/prepare_lidc_interactive_demo.py'], cwd=REPO_DIR)
manifest_path = REPO_DIR / 'data/validation/lidc_idri_0686/interactive_demo_manifest.json'
manifest = json.loads(manifest_path.read_text())
print('Remote API series ID:', manifest['api_series_id'])
print('DICOM Series UID:', manifest['series_instance_uid'])


In [ ]:
# Start one API worker, verify it locally, then open the ngrok HTTPS tunnel.
import os
import subprocess
import sys
import time
import requests
from google.colab import userdata
from pyngrok import ngrok

try:
    SERVER.terminate()
    SERVER.wait(timeout=10)
except NameError:
    pass
except Exception:
    pass
ngrok.kill()

server_env = dict(os.environ)
server_env.update({
    'DATABASE_URL': manifest['database_url'],
    'STORAGE_DIR': manifest['storage_dir'],
    'MEDSAM2_BACKEND': 'torch',
    'MEDSAM2_CHECKPOINT_PATH': str(checkpoint),
    'MEDSAM2_MODEL_CONFIG': 'configs/sam2.1_hiera_t512.yaml',
    'MEDSAM2_DEVICE': 'cuda',
    'OMP_NUM_THREADS': '2',
    'MKL_NUM_THREADS': '2',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
log_path = Path('/content/doctor_assistant_colab_api.log')
SERVER_LOG = log_path.open('w')
SERVER = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.colab_server:create_app', '--factory',
     '--host', '127.0.0.1', '--port', str(API_PORT), '--workers', '1',
     '--timeout-keep-alive', '300'],
    cwd=REPO_DIR, env=server_env, stdout=SERVER_LOG, stderr=subprocess.STDOUT,
)
for _ in range(60):
    if SERVER.poll() is not None:
        SERVER_LOG.flush()
        raise RuntimeError(log_path.read_text()[-6000:])
    try:
        local_health = requests.get(f'http://127.0.0.1:{API_PORT}/health', timeout=2)
        if local_health.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('API did not become healthy within 60 seconds')
health = local_health.json()
assert health['medsam2_configured'], health

ngrok_token = userdata.get('NGROK_TOKEN')
assert ngrok_token, 'Add NGROK_TOKEN in Colab Secrets and enable notebook access.'
ngrok.set_auth_token(ngrok_token)
TUNNEL = ngrok.connect(addr=API_PORT, proto='http')
PUBLIC_API_URL = TUNNEL.public_url.replace('http://', 'https://')
public_health = requests.get(
    f'{PUBLIC_API_URL}/health',
    headers={'ngrok-skip-browser-warning': 'doctor-assistant'},
    timeout=20,
)
public_health.raise_for_status()
print('\nCOLAB API READY:', PUBLIC_API_URL)
print('Health:', public_health.json())
print('\nRun on the Mac:')
print(f"bash scripts/start_ohif_with_remote_api.sh '{PUBLIC_API_URL}'")


In [ ]:
# Optional one-time relay when FortiGuard blocks orthanc.uclouvain.be on the Mac.
# Colab fetches the official universal package, then the browser saves it to Downloads.
from google.colab import files

if DOWNLOAD_ORTHANC_MACOS:
    orthanc_url = 'https://orthanc.uclouvain.be/downloads/macos/packages/universal/Orthanc-macOS-26.4.2.zip'
    orthanc_zip = Path('/content/Orthanc-macOS-26.4.2.zip')
    if not orthanc_zip.exists():
        print('Downloading official Orthanc macOS package in Colab (~320 MB)...', flush=True)
        with requests.get(orthanc_url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with orthanc_zip.open('wb') as output:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        output.write(chunk)
    assert orthanc_zip.stat().st_size > 300 * 1024**2, 'Orthanc download is unexpectedly small.'
    print(f'Ready: {orthanc_zip} ({orthanc_zip.stat().st_size / 1024**2:.1f} MiB)')
    files.download(str(orthanc_zip))
    print('After the Mac download finishes: bash scripts/install_orthanc_macos.sh')
else:
    print('Orthanc macOS relay skipped. Set DOWNLOAD_ORTHANC_MACOS=True to use it.')


In [ ]:
# Keep the interactive runtime active and show bounded status. Interrupt this cell to stop monitoring.
import time
print('Monitoring one-worker API. Interrupt this cell when the test is finished.')
while SERVER.poll() is None:
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f'API alive | CUDA allocated {allocated:.2f} GiB | reserved {reserved:.2f} GiB', flush=True)
    time.sleep(30)
print('API process exited. Tail of log:')
print(log_path.read_text()[-6000:])


In [ ]:
# Cleanup after interrupting the monitoring cell.
ngrok.kill()
if SERVER.poll() is None:
    SERVER.terminate()
    try:
        SERVER.wait(timeout=15)
    except subprocess.TimeoutExpired:
        SERVER.kill()
SERVER_LOG.close()
torch.cuda.empty_cache()
print('ngrok tunnel closed and API process stopped.')
